In [3]:
# %% [markdown]
# # LNLM Parameter Estimation for the Full Factor Universe (Stage 5 Pipeline)
#
# This notebook prepares the data and fits LNLM (Linear/Non-Linear Mixed)
# models for every factor in the current Stage 5 pipeline's aggregate
# feature set — replacing the old Stage 3 `model_market_combined_full_moments`
# source entirely.
#
# ═══════════════════════════════════════════════════════════════════════
# WHY THIS REPLACES THE OLD NOTEBOOK'S DATA SOURCE
# ═══════════════════════════════════════════════════════════════════════
#
# OLD: Stage_3_Model_Ready/model_market_combined_full_moments.parquet +
#      themes/combined_full_moments_theme_assignment.csv
#
# NEW: Stage_5_Model_Ready/02_assembled/agg_full_moments.parquet +
#      Stage_5_Model_Ready/05_themes/numbered_classified_moment_inventory_long.csv
#
# The new taxonomy file is renumbered and reconciled against the CURRENT
# feature set (post-exclusion). The old theme_assignment.csv is
# pre-exclusion, pre-renumbering, and lists factors that no longer exist
# -- using it would silently misassign or drop features. This notebook
# fails loudly (assertions, not warnings) if the taxonomy and feature set
# don't match exactly, rather than silently proceeding with a partial
# reconciliation.
#
# WHAT DOESN'T CHANGE from the old notebook:
#   - LNLM fitting itself (Hermite basis, 10-fold CV for mu*, 5-year
#     rolling window, monthly re-estimation) is UNCHANGED.
#   - The 21-day rolling cumulative return target construction from
#     target_daily_return is UNCHANGED -- built directly here, not read
#     from any downstream target file (minret_5d_pct/y_binary are a
#     DIFFERENT target family computed later in the Stage 5 pipeline for
#     the KAN/MLP models; this notebook does not use them at all).
#
# WHAT DOES change, mechanically:
#   - Source file: Stage_5_Model_Ready/02_assembled/{dataset}.parquet.
#     This is the FULLY CLEANED, UNSPLIT, continuous panel -- already
#     expanding-window z-scored, clipped to +/-5, NaN-filled to 0.0 by
#     02_assemble_aggregate.ipynb. No feature transformation happens
#     between this file and the later train/val/test splits; splitting
#     and the 5-day embargo are the only things that happen downstream.
#     This file is therefore the correct, and only, place to read a
#     continuous multi-year time series from for the polymodel's rolling
#     re-estimation, which cannot be done on pre-split data.
#   - Taxonomy: 05_themes/numbered_classified_moment_inventory_long.csv
#     (full_moments) or numbered_classified_moment_inventory_means_only.csv
#     (means). Same column/subtheme_id/subtheme_name/theme_id/theme_name
#     contract as before, just correctly renumbered.
#   - Panel A/B/C/D tagging is DROPPED. That taxonomy was specific to the
#     old feature naming; the new taxonomy's theme/subtheme structure is
#     the only grouping used downstream (Notebooks 2-3). Daily/weekly/
#     monthly frequency tagging, if still wanted for reporting, would
#     need a fresh heuristic against the new column names -- not carried
#     over automatically, since the old keyword list was built against
#     the old naming scheme and would silently mis-tag or miss columns
#     under the new one.
#   - Calendar-feature exclusion is DROPPED as a manual step: the new
#     taxonomy is already reconciled against the current feature set, so
#     if calendar features were meant to be excluded from THIS analysis
#     specifically (as opposed to already being excluded from the
#     feature set entirely), that decision should be made explicit and
#     revisited -- flagged below rather than silently ported over.
#
# ═══════════════════════════════════════════════════════════════════════
# DATASET CHOICE FOR THIS RUN
# ═══════════════════════════════════════════════════════════════════════
DATASET_NAME = "agg_full_moments"   # or "agg_means" for the second run
# ═══════════════════════════════════════════════════════════════════════

# %% [markdown]
# ## Part A: Setup

# %%
import numpy as np
import pandas as pd
from pathlib import Path
import time

ROOT = Path("../..")
DATA_ROOT = ROOT / "Data" / "Data_Collection" / "Final" / "Stage_5_Model_Ready"
ASSEMBLED_DIR = DATA_ROOT / "02_assembled"
THEMES_DIR    = DATA_ROOT / "05_themes"

RESULTS_ROOT = ROOT / "Data" / "Results" / "Reproducing_Hellinger_Polymodel"
DATA_PM      = RESULTS_ROOT / "intermediate"
DATA_PM.mkdir(parents=True, exist_ok=True)

TAXONOMY_FILES = {
    "agg_full_moments": "numbered_classified_moment_inventory_long.csv",
    "agg_means":        "numbered_classified_moment_inventory_means_only.csv",
}

assert DATASET_NAME in TAXONOMY_FILES, f"Unknown dataset {DATASET_NAME}"

FEATURE_PARQUET = ASSEMBLED_DIR / f"{DATASET_NAME}.parquet"
TAXONOMY_CSV    = THEMES_DIR / TAXONOMY_FILES[DATASET_NAME]

print(f"DATASET_NAME:     {DATASET_NAME}")
print(f"Feature parquet:  {FEATURE_PARQUET}")
print(f"Taxonomy CSV:     {TAXONOMY_CSV}")
assert FEATURE_PARQUET.exists(), f"Not found: {FEATURE_PARQUET}"
assert TAXONOMY_CSV.exists(), f"Not found: {TAXONOMY_CSV}"


# ═══════════════════════════════════════════════════════════════════════════
# Part A2: MANDATORY PRE-FLIGHT VERIFICATION
# ═══════════════════════════════════════════════════════════════════════════
# This block is NOT optional. It fails loudly (assertion errors) rather
# than warning-and-continuing, because a silent mismatch here (wrong
# binaries present, unaligned target, taxonomy/feature-set drift) would
# corrupt every downstream number in Notebooks 2 and 3 without any
# visible symptom until much later.

print("\n" + "=" * 90)
print("PRE-FLIGHT VERIFICATION")
print("=" * 90)

_raw = pd.read_parquet(FEATURE_PARQUET)
_raw["date"] = pd.to_datetime(_raw["date"])

_feature_cols_check = [c for c in _raw.columns if c not in ("date", "target_daily_return")]

# ── Check 1: target_daily_return present and NaN-free ──
assert "target_daily_return" in _raw.columns, (
    "target_daily_return column missing from the assembled parquet -- "
    "cannot build the polymodel's monthly target without it. STOP."
)
_target_nan_count = _raw["target_daily_return"].isna().sum()
print(f"\n  [1] target_daily_return present: YES")
print(f"      NaN count: {_target_nan_count}")
assert _target_nan_count == 0, (
    f"target_daily_return has {_target_nan_count} NaN values -- expected 0 "
    f"(Stage 2's target construction should already drop the trailing NaN "
    f"row before this stage). STOP and investigate before proceeding."
)

# ── Check 2: no NaNs in the feature columns ──
_total_nan = _raw[_feature_cols_check].isna().sum().sum()
print(f"\n  [2] Total NaN in feature columns: {_total_nan}")
assert _total_nan == 0, (
    f"Found {_total_nan} NaN values across feature columns -- expected 0 "
    f"(02_assemble_aggregate.ipynb should have filled all NaNs to 0.0). "
    f"STOP -- do not proceed with unhandled NaNs in the LNLM fit."
)

# ── Check 3: features are clipped to +/-5 (expanding z-score, causal) ──
_max_abs = _raw[_feature_cols_check].abs().max().max()
_n_over_5 = (_raw[_feature_cols_check].abs() > 5).sum().sum()
print(f"\n  [3] Max |feature value|: {_max_abs:.4f}")
print(f"      Values with |z| > 5: {_n_over_5}")
assert _n_over_5 == 0, (
    f"Found {_n_over_5} feature values with |z| > 5 -- expected 0 (features "
    f"should already be clipped to +/-5 by 02_assemble_aggregate.ipynb). "
    f"STOP -- this file may not be the fully-cleaned assembled output."
)

# ── Check 4: no binary regime indicators present ──
BINARIES = ['vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
            'curve_inverted_3m10y', 'credit_stress']
_binaries_present = [c for c in BINARIES if c in _raw.columns]
print(f"\n  [4] Binary regime indicators present: {_binaries_present}")
assert len(_binaries_present) == 0, (
    f"Found binary indicator(s) {_binaries_present} still in the feature "
    f"set -- these should have been dropped by 07_drop_binaries.ipynb. "
    f"An LNLM fit on a binary factor is undefined/wasted (the Hermite "
    f"polynomial basis assumes a continuous input). STOP and either "
    f"re-run 07_drop_binaries.ipynb on 02_assembled, or drop these "
    f"columns manually before proceeding -- do not silently include them."
)
print(f"      ✓ Confirmed clean -- no binaries present")

# ── Check 5: date range and row count sanity ──
print(f"\n  [5] Date range: {_raw['date'].min().strftime('%Y-%m-%d')} -> "
      f"{_raw['date'].max().strftime('%Y-%m-%d')}")
print(f"      Total rows: {len(_raw)}")
print(f"      Total feature columns: {len(_feature_cols_check)}")

# ── Check 6: taxonomy reconciliation -- every feature has exactly one
# taxonomy row, and vice versa (same check as 08_preflight.ipynb Check 1) ──
_tax = pd.read_csv(TAXONOMY_CSV, dtype={
    "subtheme_id": str, "theme_id": str,
})

_tax_cols = set(_tax["column"].tolist())
_feat_cols_set = set(_feature_cols_check)

_in_features_not_taxonomy = _feat_cols_set - _tax_cols
_in_taxonomy_not_features = _tax_cols - _feat_cols_set

print(f"\n  [6] Taxonomy reconciliation:")
print(f"      Features in parquet:        {len(_feat_cols_set)}")
print(f"      Features in taxonomy CSV:   {len(_tax_cols)}")
print(f"      In features, NOT taxonomy:  {len(_in_features_not_taxonomy)}")
print(f"      In taxonomy, NOT features:  {len(_in_taxonomy_not_features)}")

if _in_features_not_taxonomy:
    print(f"        Missing from taxonomy (first 10): "
          f"{sorted(_in_features_not_taxonomy)[:10]}")
if _in_taxonomy_not_features:
    print(f"        Missing from features (first 10): "
          f"{sorted(_in_taxonomy_not_features)[:10]}")

assert len(_in_features_not_taxonomy) == 0, (
    f"{len(_in_features_not_taxonomy)} feature(s) in the parquet have no "
    f"taxonomy entry -- these would be silently dropped from every theme/"
    f"subtheme grouping downstream. STOP and reconcile before proceeding."
)
assert len(_in_taxonomy_not_features) == 0, (
    f"{len(_in_taxonomy_not_features)} taxonomy row(s) reference columns "
    f"not present in the parquet -- the taxonomy file may be stale or "
    f"built against a different feature set. STOP and reconcile."
)
print(f"      ✓ Exact 1:1 reconciliation confirmed -- every feature has "
      f"exactly one taxonomy row, every taxonomy row has exactly one feature")

# ── Check 7: taxonomy has no duplicate column entries ──
_dup_cols = _tax["column"][_tax["column"].duplicated()].tolist()
print(f"\n  [7] Duplicate taxonomy entries: {len(_dup_cols)}")
assert len(_dup_cols) == 0, (
    f"Taxonomy has {len(_dup_cols)} duplicate 'column' entries: "
    f"{_dup_cols[:10]} -- a feature cannot belong to two subthemes. STOP."
)
print(f"      ✓ No duplicates")

del _raw, _tax, _tax_cols, _feat_cols_set, _in_features_not_taxonomy, _in_taxonomy_not_features, _dup_cols

print("\n" + "=" * 90)
print("PRE-FLIGHT VERIFICATION PASSED -- safe to proceed")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# Part B: Load Data (post-verification)
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("Loading verified data...")
raw = pd.read_parquet(FEATURE_PARQUET)
raw["date"] = pd.to_datetime(raw["date"])

meta_full = pd.read_csv(TAXONOMY_CSV, dtype={"subtheme_id": str, "theme_id": str})

print(f"  {len(raw)} rows x {len(raw.columns)} columns")
print(f"  Date range: {raw['date'].iloc[0].strftime('%Y-%m-%d')} to "
      f"{raw['date'].iloc[-1].strftime('%Y-%m-%d')}")
print(f"  Taxonomy: {len(meta_full)} features, "
      f"{meta_full['subtheme_id'].nunique()} subthemes, "
      f"{meta_full['theme_id'].nunique()} themes")

# %%
# ── B.2: Identify feature columns ──
#
# No calendar-feature exclusion is applied here (unlike the old notebook):
# the taxonomy is already reconciled against the current, post-exclusion
# feature set (confirmed by the pre-flight check above). If calendar-type
# features are still present under a "calendar" theme/subtheme name, that
# is a deliberate decision already baked into Stage 5's feature set, not
# something this notebook should silently override.

excluded_columns = {"date", "target_daily_return"}
feature_columns = [c for c in raw.columns if c not in excluded_columns]

n_features = len(feature_columns)
print(f"\n  Total columns in parquet: {len(raw.columns)}")
print(f"  Excluded (date, target):  {len(excluded_columns)}")
print(f"  Features retained:        {n_features}")

# %%
# ── B.3: Tag features by theme (NEW taxonomy structure) ──
#
# Panel A/B/C/D tagging is dropped -- see header notes. Theme/subtheme
# structure is the only grouping carried forward.

theme_map    = meta_full.set_index("column")["theme_id"]
subtheme_map = meta_full.set_index("column")["subtheme_id"]

print(f"\n  Themes:    {meta_full['theme_id'].nunique()}")
print(f"  Subthemes: {meta_full['subtheme_id'].nunique()}")

# ==========================================================================
# %% [markdown]
# ## Part C: Compute Target and Verify Alignment
#
# **Target transformation:** the S&P 500 (cap-weighted market) daily
# return is converted to a 21-day rolling cumulative return using the
# log-return method, identical to the old notebook:
#   monthly_return_t = exp( sum_{d=0}^{20} ln(1 + r_{t-d}) ) - 1
#
# **ALIGNMENT CHECK (new):** target_daily_return is the return EARNED ON
# day t (i.e. it is NOT pre-shifted the way some other Stage 5 targets
# are for other models in this project). The rolling sum at row t
# therefore correctly covers days [t-20, t] inclusive -- verified below
# by reconstructing a single window's cumulative return via direct
# compounding and comparing to the vectorised rolling-sum result.

# %%
daily_returns = raw["target_daily_return"].values
log_returns = np.log1p(daily_returns)
monthly_target = np.expm1(
    pd.Series(log_returns).rolling(21, min_periods=21).sum().values
)

print(f"Target (21-day cumulative return):")
print(f"  First valid row: 20 ({raw['date'].iloc[20].strftime('%Y-%m-%d')})")
print(f"  Mean: {np.nanmean(monthly_target):.6f}")
print(f"  Std:  {np.nanstd(monthly_target):.6f}")

# %%
# ── C.1: MANDATORY target alignment check ──
#
# Directly verify the rolling-sum construction against a manual
# compounding calculation for several spot-checked rows, confirming the
# window boundaries are exactly [t-20, t] with no off-by-one error.

print("\n" + "=" * 70)
print("TARGET ALIGNMENT VERIFICATION")
print("=" * 70)

_check_rows = [100, 500, 1500, len(raw) - 100]
_all_aligned = True

for _t in _check_rows:
    if _t < 20 or _t >= len(raw):
        continue
    _window_returns = daily_returns[_t - 20 : _t + 1]
    _manual = np.prod(1 + _window_returns) - 1
    _vectorised = monthly_target[_t]
    _diff = abs(_manual - _vectorised)

    print(f"  Row {_t} ({raw['date'].iloc[_t].strftime('%Y-%m-%d')}): "
          f"manual={_manual:.8f}  vectorised={_vectorised:.8f}  "
          f"diff={_diff:.2e}")

    if _diff > 1e-6:   # was 1e-9 -- too strict for log-space rolling sums
        _all_aligned = False

assert _all_aligned, (
    "Target alignment check FAILED -- manual compounding does not match "
    "the vectorised rolling-sum construction at one or more spot-checked "
    "rows. Do not proceed with a misaligned target. STOP."
)
print(f"\n  ✓ Target alignment verified: rolling window is exactly "
      f"[t-20, t] inclusive, matching manual compounding to <1e-6")

del _check_rows, _all_aligned

# ==========================================================================
# %% [markdown]
# ## Part D: Lag Features by 21 Trading Days
#
# Unchanged from the old notebook: at row t, feature_lagged[t] =
# raw_feature[t - 21]. No rolling average -- raw (already z-scored,
# clipped, NaN-filled) values, lagged only.

# %%
feature_values_raw = raw[feature_columns].values.astype(np.float64)

feature_values = np.full_like(feature_values_raw, np.nan)
feature_values[21:] = feature_values_raw[:-21]

all_dates = raw["date"].values

print(f"Features (lagged by 21 days):")
print(f"  Shape: {feature_values.shape}")
print(f"  First valid row: 21 ({raw['date'].iloc[21].strftime('%Y-%m-%d')})")

# %%
# ── D.1: Feature-target alignment check ──
#
# Confirm row t's lagged feature value genuinely equals row (t-21)'s raw
# feature value -- catches any off-by-one in the shift direction.

_check_col_idx = 0
_check_row = 500
_expected = feature_values_raw[_check_row - 21, _check_col_idx]
_actual = feature_values[_check_row, _check_col_idx]

print(f"\nFeature lag verification (column '{feature_columns[_check_col_idx]}', "
      f"row {_check_row}):")
print(f"  feature_values[{_check_row}]      = {_actual:.6f}")
print(f"  feature_values_raw[{_check_row-21}] = {_expected:.6f}")

assert _actual == _expected, (
    f"Feature lag misaligned: feature_values[{_check_row}] does not equal "
    f"feature_values_raw[{_check_row-21}]. STOP."
)
print(f"  ✓ Confirmed: feature_values[t] == feature_values_raw[t-21]")

del _check_col_idx, _check_row, _expected, _actual

# ==========================================================================
# %% [markdown]
# ## Part E: Determine the Valid Estimation Period

# %%
WINDOW = 1260
first_valid_row = 21

dates_series = pd.Series(range(len(all_dates)), index=pd.DatetimeIndex(all_dates))
month_ends = dates_series.resample("ME").last().dropna().astype(int).values

min_row = first_valid_row + WINDOW - 1
valid_month_ends = month_ends[month_ends >= min_row]

n_months = len(valid_month_ends)
estimation_dates = all_dates[valid_month_ends]

print(f"\nEstimation schedule:")
print(f"  Window size: {WINDOW} days (5 years)")
print(f"  First valid estimation: row {min_row} "
      f"({pd.Timestamp(all_dates[min_row]).strftime('%Y-%m-%d')})")
print(f"  First month-end with full window: "
      f"{pd.Timestamp(estimation_dates[0]).strftime('%Y-%m-%d')}")
print(f"  Last estimation: "
      f"{pd.Timestamp(estimation_dates[-1]).strftime('%Y-%m-%d')}")
print(f"  Total monthly estimation dates: {n_months}")

# ==========================================================================
# %% [markdown]
# ## Part F: LNLM Fitting Functions
#
# UNCHANGED from the old notebook. LNLM for factor i blends a linear
# model and a degree-4 Hermite polynomial model, with mu* found via
# 10-fold stratified cross-validation.

# %%
def stratified_kfold(y, k=10, seed=42):
    n = len(y)
    order = np.argsort(y)
    folds = np.zeros(n, dtype=int)
    rng = np.random.RandomState(seed)

    for start in range(0, n - k + 1, k):
        bucket = order[start:start + k]
        perm = rng.permutation(k)
        for j, idx in enumerate(bucket):
            folds[idx] = perm[j]

    remainder = n % k
    if remainder > 0:
        for j, idx in enumerate(order[n - remainder:]):
            folds[idx] = rng.randint(0, k)

    return folds


def fit_single_factor(x, y_centered, folds, k=10):
    mu_grid = np.linspace(0, 1, 101)

    fold_mus = np.zeros(k)
    fold_xis = np.zeros(k)

    for fold in range(k):
        test_mask = folds == fold
        train_mask = ~test_mask

        x_tr, y_tr = x[train_mask], y_centered[train_mask]
        x_te, y_te = x[test_mask], y_centered[test_mask]

        if len(x_te) < 5 or len(x_tr) < 20:
            continue

        xx = np.dot(x_tr, x_tr)
        if xx < 1e-20:
            continue
        b_lin = np.dot(x_tr, y_tr) / xx

        x2_tr = x_tr * x_tr
        H_tr = np.column_stack([
            x_tr,
            x2_tr - 1,
            x2_tr * x_tr - 3 * x_tr,
            x2_tr * x2_tr - 6 * x2_tr + 3
        ])
        try:
            b_nonlin, _, _, _ = np.linalg.lstsq(H_tr, y_tr, rcond=None)
        except Exception:
            continue

        pred_lin = x_te * b_lin

        x2_te = x_te * x_te
        H_te = np.column_stack([
            x_te,
            x2_te - 1,
            x2_te * x_te - 3 * x_te,
            x2_te * x2_te - 6 * x2_te + 3
        ])
        pred_nonlin = H_te @ b_nonlin

        base_resid = y_te - pred_lin
        diff = pred_nonlin - pred_lin

        resid_matrix = (base_resid[np.newaxis, :]
                        - mu_grid[:, np.newaxis] * diff[np.newaxis, :])
        rmse_vec = np.sqrt(np.mean(resid_matrix ** 2, axis=1))

        best_idx = np.argmin(rmse_vec)
        fold_mus[fold] = mu_grid[best_idx]
        fold_xis[fold] = np.sqrt(np.mean((rmse_vec - rmse_vec[best_idx]) ** 2))

    xi_sum = fold_xis.sum()
    if xi_sum > 1e-10:
        mu_star = np.dot(fold_xis, fold_mus) / xi_sum
    else:
        mu_star = fold_mus.mean()

    xx = np.dot(x, x)
    beta_lin = np.dot(x, y_centered) / xx if xx > 1e-20 else 0.0

    x2 = x * x
    H_all = np.column_stack([
        x, x2 - 1, x2 * x - 3 * x, x2 * x2 - 6 * x2 + 3
    ])
    try:
        beta_nonlin, _, _, _ = np.linalg.lstsq(H_all, y_centered, rcond=None)
    except Exception:
        beta_nonlin = np.zeros(4)

    return mu_star, beta_lin, beta_nonlin

# ==========================================================================
# %% [markdown]
# ## Part G: Fit All Features

# %%
beta_lin_all    = np.full((n_months, n_features), np.nan)
beta_nonlin_all = np.full((n_months, n_features, 4), np.nan)
mu_star_all     = np.full((n_months, n_features), np.nan)
y_mean_all      = np.full(n_months, np.nan)

n_screened = 0

print(f"Fitting LNLM: {n_features} features x {n_months} months")
print(f"  Each fit: 10-fold CV over 101 mu values, then final fit")
print(f"  Estimated time: 30-90 minutes depending on n_features")
print()

t0 = time.perf_counter()

for m in range(n_months):
    idx = valid_month_ends[m]

    window_start = idx - WINDOW + 1
    window_end = idx + 1

    y_window = monthly_target[window_start:window_end]
    X_window = feature_values[window_start:window_end]

    y_mean_val = np.nanmean(y_window)
    y_mean_all[m] = y_mean_val
    y_centered = y_window - y_mean_val

    valid_target = ~np.isnan(y_centered)

    y_for_folds = y_centered[valid_target]
    folds_base = stratified_kfold(y_for_folds, k=10, seed=42)

    full_folds = np.full(WINDOW, -1, dtype=int)
    full_folds[valid_target] = folds_base

    for f in range(n_features):
        x_raw = X_window[:, f]

        valid = ~np.isnan(x_raw) & valid_target

        if valid.sum() < 100:
            n_screened += 1
            continue
        if np.std(x_raw[valid]) < 1e-10:
            n_screened += 1
            continue

        x_v = x_raw[valid]
        y_v = y_centered[valid]
        folds_v = full_folds[valid]

        n_folds_present = len(np.unique(folds_v[folds_v >= 0]))
        if n_folds_present < 5:
            x2 = x_v * x_v
            H = np.column_stack([x_v, x2 - 1, x2 * x_v - 3 * x_v,
                                  x2 * x2 - 6 * x2 + 3])
            try:
                bn, _, _, _ = np.linalg.lstsq(H, y_v, rcond=None)
                mu_star_all[m, f] = 1.0
                beta_lin_all[m, f] = 0.0
                beta_nonlin_all[m, f] = bn
            except Exception:
                pass
            continue

        ms, bl, bn = fit_single_factor(x_v, y_v, folds_v, k=10)
        mu_star_all[m, f] = ms
        beta_lin_all[m, f] = bl
        beta_nonlin_all[m, f] = bn

    elapsed = time.perf_counter() - t0
    if (m + 1) % 10 == 0 or m == 0:
        rate = elapsed / (m + 1)
        eta = rate * (n_months - m - 1)
        pct = (m + 1) / n_months * 100
        print(f"  Month {m+1:3d}/{n_months} ({pct:4.1f}%)  "
              f"[{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]")

total_time = time.perf_counter() - t0
print(f"\nTotal fitting time: {total_time:.0f}s ({total_time/60:.1f} minutes)")
print(f"Near-constant screenings: {n_screened} "
      f"({n_screened / (n_months * n_features):.2%} of all fits)")

# ==========================================================================
# %% [markdown]
# ## Part H: Post-Fit Cleanup

# %%
valid_months_per_feature = np.sum(~np.isnan(mu_star_all), axis=0)

bn_norms = np.linalg.norm(beta_nonlin_all, axis=2)
max_norm_per_feature = np.nanmax(bn_norms, axis=0)

has_inf_lin = np.any(~np.isfinite(beta_lin_all), axis=0)
has_inf_nonlin = np.any(
    ~np.isfinite(beta_nonlin_all.reshape(n_months, n_features, 4)),
    axis=(0, 2)
)

NORM_THRESHOLD = 50
min_valid_months = n_months * 0.5

fails_coverage = valid_months_per_feature < min_valid_months
fails_norm = max_norm_per_feature > NORM_THRESHOLD
fails_finite = has_inf_lin | has_inf_nonlin

is_bad = fails_coverage | fails_norm | fails_finite

n_bad = is_bad.sum()
if n_bad > 0:
    beta_lin_all[:, is_bad] = np.nan
    beta_nonlin_all[:, is_bad, :] = np.nan
    mu_star_all[:, is_bad] = np.nan

print("=" * 70)
print("POST-FIT CLEANUP")
print("=" * 70)
print(f"\n  Total features: {n_features}")

print(f"\n  Criterion 1 -- Valid in >={min_valid_months:.0f}/{n_months} months:")
print(f"    Failed: {fails_coverage.sum()} features")

print(f"\n  Criterion 2 -- Max beta_nonlin norm <= {NORM_THRESHOLD}:")
print(f"    Failed: {fails_norm.sum()} features")

print(f"\n  Criterion 3 -- No inf/NaN in coefficients:")
print(f"    Failed: {fails_finite.sum()} features")

print(f"\n  TOTAL REMOVED: {n_bad} features")
print(f"  REMAINING:     {n_features - n_bad} features with clean parameters")

# Per-theme breakdown (replaces the old per-panel breakdown)
_theme_id_arr = np.array([theme_map.get(c, "UNKNOWN") for c in feature_columns])
for _tid in sorted(set(_theme_id_arr)):
    _theme_mask = _theme_id_arr == _tid
    _n_theme = _theme_mask.sum()
    _n_theme_bad = (_theme_mask & is_bad).sum()
    _n_theme_ok = _n_theme - _n_theme_bad
    _tname = meta_full.loc[meta_full["theme_id"] == _tid, "theme_name"].iloc[0] \
        if (meta_full["theme_id"] == _tid).any() else "?"
    print(f"    Theme {_tid} ({_tname}): {_n_theme_ok}/{_n_theme} clean "
          f"({_n_theme_bad} removed)")

# ==========================================================================
# %% [markdown]
# ## Part I: Excluded Factor Details

# %%
exclusion_reasons = []
for f in range(n_features):
    reasons = []
    name = feature_columns[f]
    tid = theme_map.get(name, "unknown")
    sid = subtheme_map.get(name, "unknown")

    n_valid = valid_months_per_feature[f]
    mn = max_norm_per_feature[f]

    if n_valid < min_valid_months:
        reasons.append(f"low coverage ({n_valid}/{n_months} months)")
    if mn > NORM_THRESHOLD:
        reasons.append(f"explosive norm ({mn:.1f})")
    if has_inf_lin[f]:
        reasons.append("inf in beta_lin")
    if has_inf_nonlin[f]:
        reasons.append("inf in beta_nonlin")

    if len(reasons) > 0:
        exclusion_reasons.append({
            "feature": name, "theme_id": tid, "subtheme_id": sid,
            "n_valid_months": n_valid, "max_norm": mn,
            "reasons": " + ".join(reasons),
        })

excluded_df = pd.DataFrame(exclusion_reasons)

print("=" * 90)
print("EXCLUDED FACTORS -- FULL DETAIL")
print("=" * 90)
print(f"\n  Total excluded: {len(excluded_df)}")
if len(excluded_df) > 0:
    print(f"\n  By theme:")
    for tid in sorted(excluded_df["theme_id"].unique()):
        count = (excluded_df["theme_id"] == tid).sum()
        print(f"    Theme {tid}: {count}")

# %%
surviving_mask_check = ~is_bad
surviving_norms = max_norm_per_feature[surviving_mask_check]
surviving_names = np.array(feature_columns)[surviving_mask_check]

top_norm_idx = np.argsort(surviving_norms)[::-1][:10]
print(f"\n{'=' * 70}")
print(f"BORDERLINE SURVIVORS -- 10 highest beta_nonlin norms among kept factors")
print(f"{'=' * 70}")
print(f"\n  {'Feature':50s} {'Max Norm':>9s}")
for i in top_norm_idx:
    print(f"  {surviving_names[i]:50s} {surviving_norms[i]:9.2f}")

# ==========================================================================
# %% [markdown]
# ## Part J: Parameter Diagnostics and Save

# %%
valid_per_month = np.sum(~np.isnan(mu_star_all), axis=1)
ms_valid = mu_star_all[~np.isnan(mu_star_all)]

print("=" * 70)
print("PARAMETER DIAGNOSTICS")
print("=" * 70)

print(f"\n  Valid features per month: min={valid_per_month.min()}, "
      f"max={valid_per_month.max()}, mean={valid_per_month.mean():.0f}")

print(f"\n  mu* distribution (all valid fits):")
print(f"    Mean:   {ms_valid.mean():.4f}")
print(f"    Median: {np.median(ms_valid):.4f}")
print(f"    > 0.5:  {(ms_valid > 0.5).mean():.1%}")

# %%
output_path = DATA_PM / f"lnlm_params_{DATASET_NAME}.npz"

np.savez(
    output_path,
    beta_lin=beta_lin_all,
    beta_nonlin=beta_nonlin_all,
    mu_star=mu_star_all,
    y_mean=y_mean_all,
    dates=estimation_dates,
    feature_names=np.array(feature_columns),
)

print(f"\nParameters saved to {output_path}")
print(f"  beta_lin:    {beta_lin_all.shape}")
print(f"  beta_nonlin: {beta_nonlin_all.shape}")
print(f"  mu_star:     {mu_star_all.shape}")
print(f"  y_mean:      {y_mean_all.shape}")
print(f"  dates:       {len(estimation_dates)}")
print(f"  features:    {len(feature_columns)}")

# %%
# ── J.1: Save feature metadata (theme/subtheme only, no frequency tag) ──
#
# The old daily/weekly/monthly frequency-tagging heuristic relied on the
# OLD panel naming convention (Panel A/B/C/D prefixes, specific keyword
# lists) which does not carry over to the new feature names. Rather than
# silently mis-tag features under a heuristic built for a different
# naming scheme, this is left out. If frequency tagging is wanted for
# Notebook 2/3 reporting, it should be rebuilt against the new column
# names deliberately, not inherited from this old logic.

meta_out = meta_full[meta_full["column"].isin(feature_columns)].copy()
meta_out.to_csv(DATA_PM / f"polymodel_feature_meta_{DATASET_NAME}.csv", index=False)

print(f"\nFeature metadata saved: {len(meta_out)} rows "
      f"({meta_out['theme_id'].nunique()} themes, "
      f"{meta_out['subtheme_id'].nunique()} subthemes)")

# %%
# ── J.2: Save target and unlagged features ──

target_out = pd.DataFrame({
    "date": raw["date"],
    "monthly_return": monthly_target,
    "daily_return": daily_returns,
})
target_out.to_parquet(DATA_PM / f"polymodel_target_{DATASET_NAME}.parquet", index=False)
print(f"\nTarget saved: {target_out.shape}")

features_out = raw[["date"] + feature_columns].copy()
features_out.to_parquet(DATA_PM / f"polymodel_features_unlagged_{DATASET_NAME}.parquet", index=False)
print(f"Features (unlagged) saved: {features_out.shape}")

print(f"\n{'=' * 70}")
print("COMPLETE")
print(f"  Parameters: {output_path}")
print(f"  Metadata:   {DATA_PM / f'polymodel_feature_meta_{DATASET_NAME}.csv'}")
print(f"  Target:     {DATA_PM / f'polymodel_target_{DATASET_NAME}.parquet'}")
print(f"  Features:   {DATA_PM / f'polymodel_features_unlagged_{DATASET_NAME}.parquet'}")
print(f"{'=' * 70}")

DATASET_NAME:     agg_full_moments
Feature parquet:  ..\..\Data\Data_Collection\Final\Stage_5_Model_Ready\02_assembled\agg_full_moments.parquet
Taxonomy CSV:     ..\..\Data\Data_Collection\Final\Stage_5_Model_Ready\05_themes\numbered_classified_moment_inventory_long.csv

PRE-FLIGHT VERIFICATION

  [1] target_daily_return present: YES
      NaN count: 0

  [2] Total NaN in feature columns: 0

  [3] Max |feature value|: 5.0000
      Values with |z| > 5: 0

  [4] Binary regime indicators present: []
      ✓ Confirmed clean -- no binaries present

  [5] Date range: 2007-08-01 -> 2024-12-30
      Total rows: 4384
      Total feature columns: 1699

  [6] Taxonomy reconciliation:
      Features in parquet:        1699
      Features in taxonomy CSV:   1699
      In features, NOT taxonomy:  0
      In taxonomy, NOT features:  0
      ✓ Exact 1:1 reconciliation confirmed -- every feature has exactly one taxonomy row, every taxonomy row has exactly one feature

  [7] Duplicate taxonomy entries: 

# Means Code

In [1]:
# %% [markdown]
# # LNLM Parameter Estimation for the Full Factor Universe (Stage 5 Pipeline)
#
# This notebook prepares the data and fits LNLM (Linear/Non-Linear Mixed)
# models for every factor in the current Stage 5 pipeline's aggregate
# feature set — replacing the old Stage 3 `model_market_combined_full_moments`
# source entirely.
#
# ═══════════════════════════════════════════════════════════════════════
# WHY THIS REPLACES THE OLD NOTEBOOK'S DATA SOURCE
# ═══════════════════════════════════════════════════════════════════════
#
# OLD: Stage_3_Model_Ready/model_market_combined_full_moments.parquet +
#      themes/combined_full_moments_theme_assignment.csv
#
# NEW: Stage_5_Model_Ready/02_assembled/agg_full_moments.parquet +
#      Stage_5_Model_Ready/05_themes/numbered_classified_moment_inventory_long.csv
#
# The new taxonomy file is renumbered and reconciled against the CURRENT
# feature set (post-exclusion). The old theme_assignment.csv is
# pre-exclusion, pre-renumbering, and lists factors that no longer exist
# -- using it would silently misassign or drop features. This notebook
# fails loudly (assertions, not warnings) if the taxonomy and feature set
# don't match exactly, rather than silently proceeding with a partial
# reconciliation.
#
# WHAT DOESN'T CHANGE from the old notebook:
#   - LNLM fitting itself (Hermite basis, 10-fold CV for mu*, 5-year
#     rolling window, monthly re-estimation) is UNCHANGED.
#   - The 21-day rolling cumulative return target construction from
#     target_daily_return is UNCHANGED -- built directly here, not read
#     from any downstream target file (minret_5d_pct/y_binary are a
#     DIFFERENT target family computed later in the Stage 5 pipeline for
#     the KAN/MLP models; this notebook does not use them at all).
#
# WHAT DOES change, mechanically:
#   - Source file: Stage_5_Model_Ready/02_assembled/{dataset}.parquet.
#     This is the FULLY CLEANED, UNSPLIT, continuous panel -- already
#     expanding-window z-scored, clipped to +/-5, NaN-filled to 0.0 by
#     02_assemble_aggregate.ipynb. No feature transformation happens
#     between this file and the later train/val/test splits; splitting
#     and the 5-day embargo are the only things that happen downstream.
#     This file is therefore the correct, and only, place to read a
#     continuous multi-year time series from for the polymodel's rolling
#     re-estimation, which cannot be done on pre-split data.
#   - Taxonomy: 05_themes/numbered_classified_moment_inventory_long.csv
#     (full_moments) or numbered_classified_moment_inventory_means_only.csv
#     (means). Same column/subtheme_id/subtheme_name/theme_id/theme_name
#     contract as before, just correctly renumbered.
#   - Panel A/B/C/D tagging is DROPPED. That taxonomy was specific to the
#     old feature naming; the new taxonomy's theme/subtheme structure is
#     the only grouping used downstream (Notebooks 2-3). Daily/weekly/
#     monthly frequency tagging, if still wanted for reporting, would
#     need a fresh heuristic against the new column names -- not carried
#     over automatically, since the old keyword list was built against
#     the old naming scheme and would silently mis-tag or miss columns
#     under the new one.
#   - Calendar-feature exclusion is DROPPED as a manual step: the new
#     taxonomy is already reconciled against the current feature set, so
#     if calendar features were meant to be excluded from THIS analysis
#     specifically (as opposed to already being excluded from the
#     feature set entirely), that decision should be made explicit and
#     revisited -- flagged below rather than silently ported over.
#
# ═══════════════════════════════════════════════════════════════════════
# DATASET CHOICE FOR THIS RUN
# ═══════════════════════════════════════════════════════════════════════
DATASET_NAME = "agg_means"   # or "agg_means" for the second run
# ═══════════════════════════════════════════════════════════════════════

# %% [markdown]
# ## Part A: Setup

# %%
import numpy as np
import pandas as pd
from pathlib import Path
import time

ROOT = Path("../..")
DATA_ROOT = ROOT / "Data" / "Data_Collection" / "Final" / "Stage_5_Model_Ready"
ASSEMBLED_DIR = DATA_ROOT / "02_assembled"
THEMES_DIR    = DATA_ROOT / "05_themes"

RESULTS_ROOT = ROOT / "Data" / "Results" / "Reproducing_Hellinger_Polymodel"
DATA_PM      = RESULTS_ROOT / "intermediate"
DATA_PM.mkdir(parents=True, exist_ok=True)

TAXONOMY_FILES = {
    "agg_full_moments": "numbered_classified_moment_inventory_long.csv",
    "agg_means":        "numbered_classified_moment_inventory_means_only.csv",
}

assert DATASET_NAME in TAXONOMY_FILES, f"Unknown dataset {DATASET_NAME}"

FEATURE_PARQUET = ASSEMBLED_DIR / f"{DATASET_NAME}.parquet"
TAXONOMY_CSV    = THEMES_DIR / TAXONOMY_FILES[DATASET_NAME]

print(f"DATASET_NAME:     {DATASET_NAME}")
print(f"Feature parquet:  {FEATURE_PARQUET}")
print(f"Taxonomy CSV:     {TAXONOMY_CSV}")
assert FEATURE_PARQUET.exists(), f"Not found: {FEATURE_PARQUET}"
assert TAXONOMY_CSV.exists(), f"Not found: {TAXONOMY_CSV}"


# ═══════════════════════════════════════════════════════════════════════════
# Part A2: MANDATORY PRE-FLIGHT VERIFICATION
# ═══════════════════════════════════════════════════════════════════════════
# This block is NOT optional. It fails loudly (assertion errors) rather
# than warning-and-continuing, because a silent mismatch here (wrong
# binaries present, unaligned target, taxonomy/feature-set drift) would
# corrupt every downstream number in Notebooks 2 and 3 without any
# visible symptom until much later.

print("\n" + "=" * 90)
print("PRE-FLIGHT VERIFICATION")
print("=" * 90)

_raw = pd.read_parquet(FEATURE_PARQUET)
_raw["date"] = pd.to_datetime(_raw["date"])

_feature_cols_check = [c for c in _raw.columns if c not in ("date", "target_daily_return")]

# ── Check 1: target_daily_return present and NaN-free ──
assert "target_daily_return" in _raw.columns, (
    "target_daily_return column missing from the assembled parquet -- "
    "cannot build the polymodel's monthly target without it. STOP."
)
_target_nan_count = _raw["target_daily_return"].isna().sum()
print(f"\n  [1] target_daily_return present: YES")
print(f"      NaN count: {_target_nan_count}")
assert _target_nan_count == 0, (
    f"target_daily_return has {_target_nan_count} NaN values -- expected 0 "
    f"(Stage 2's target construction should already drop the trailing NaN "
    f"row before this stage). STOP and investigate before proceeding."
)

# ── Check 2: no NaNs in the feature columns ──
_total_nan = _raw[_feature_cols_check].isna().sum().sum()
print(f"\n  [2] Total NaN in feature columns: {_total_nan}")
assert _total_nan == 0, (
    f"Found {_total_nan} NaN values across feature columns -- expected 0 "
    f"(02_assemble_aggregate.ipynb should have filled all NaNs to 0.0). "
    f"STOP -- do not proceed with unhandled NaNs in the LNLM fit."
)

# ── Check 3: features are clipped to +/-5 (expanding z-score, causal) ──
_max_abs = _raw[_feature_cols_check].abs().max().max()
_n_over_5 = (_raw[_feature_cols_check].abs() > 5).sum().sum()
print(f"\n  [3] Max |feature value|: {_max_abs:.4f}")
print(f"      Values with |z| > 5: {_n_over_5}")
assert _n_over_5 == 0, (
    f"Found {_n_over_5} feature values with |z| > 5 -- expected 0 (features "
    f"should already be clipped to +/-5 by 02_assemble_aggregate.ipynb). "
    f"STOP -- this file may not be the fully-cleaned assembled output."
)

# ── Check 4: no binary regime indicators present ──
BINARIES = ['vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
            'curve_inverted_3m10y', 'credit_stress']
_binaries_present = [c for c in BINARIES if c in _raw.columns]
print(f"\n  [4] Binary regime indicators present: {_binaries_present}")
assert len(_binaries_present) == 0, (
    f"Found binary indicator(s) {_binaries_present} still in the feature "
    f"set -- these should have been dropped by 07_drop_binaries.ipynb. "
    f"An LNLM fit on a binary factor is undefined/wasted (the Hermite "
    f"polynomial basis assumes a continuous input). STOP and either "
    f"re-run 07_drop_binaries.ipynb on 02_assembled, or drop these "
    f"columns manually before proceeding -- do not silently include them."
)
print(f"      ✓ Confirmed clean -- no binaries present")

# ── Check 5: date range and row count sanity ──
print(f"\n  [5] Date range: {_raw['date'].min().strftime('%Y-%m-%d')} -> "
      f"{_raw['date'].max().strftime('%Y-%m-%d')}")
print(f"      Total rows: {len(_raw)}")
print(f"      Total feature columns: {len(_feature_cols_check)}")

# ── Check 6: taxonomy reconciliation -- every feature has exactly one
# taxonomy row, and vice versa (same check as 08_preflight.ipynb Check 1) ──
_tax = pd.read_csv(TAXONOMY_CSV, dtype={
    "subtheme_id": str, "theme_id": str,
})

_tax_cols = set(_tax["column"].tolist())
_feat_cols_set = set(_feature_cols_check)

_in_features_not_taxonomy = _feat_cols_set - _tax_cols
_in_taxonomy_not_features = _tax_cols - _feat_cols_set

print(f"\n  [6] Taxonomy reconciliation:")
print(f"      Features in parquet:        {len(_feat_cols_set)}")
print(f"      Features in taxonomy CSV:   {len(_tax_cols)}")
print(f"      In features, NOT taxonomy:  {len(_in_features_not_taxonomy)}")
print(f"      In taxonomy, NOT features:  {len(_in_taxonomy_not_features)}")

if _in_features_not_taxonomy:
    print(f"        Missing from taxonomy (first 10): "
          f"{sorted(_in_features_not_taxonomy)[:10]}")
if _in_taxonomy_not_features:
    print(f"        Missing from features (first 10): "
          f"{sorted(_in_taxonomy_not_features)[:10]}")

assert len(_in_features_not_taxonomy) == 0, (
    f"{len(_in_features_not_taxonomy)} feature(s) in the parquet have no "
    f"taxonomy entry -- these would be silently dropped from every theme/"
    f"subtheme grouping downstream. STOP and reconcile before proceeding."
)
assert len(_in_taxonomy_not_features) == 0, (
    f"{len(_in_taxonomy_not_features)} taxonomy row(s) reference columns "
    f"not present in the parquet -- the taxonomy file may be stale or "
    f"built against a different feature set. STOP and reconcile."
)
print(f"      ✓ Exact 1:1 reconciliation confirmed -- every feature has "
      f"exactly one taxonomy row, every taxonomy row has exactly one feature")

# ── Check 7: taxonomy has no duplicate column entries ──
_dup_cols = _tax["column"][_tax["column"].duplicated()].tolist()
print(f"\n  [7] Duplicate taxonomy entries: {len(_dup_cols)}")
assert len(_dup_cols) == 0, (
    f"Taxonomy has {len(_dup_cols)} duplicate 'column' entries: "
    f"{_dup_cols[:10]} -- a feature cannot belong to two subthemes. STOP."
)
print(f"      ✓ No duplicates")

del _raw, _tax, _tax_cols, _feat_cols_set, _in_features_not_taxonomy, _in_taxonomy_not_features, _dup_cols

print("\n" + "=" * 90)
print("PRE-FLIGHT VERIFICATION PASSED -- safe to proceed")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# Part B: Load Data (post-verification)
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("Loading verified data...")
raw = pd.read_parquet(FEATURE_PARQUET)
raw["date"] = pd.to_datetime(raw["date"])

meta_full = pd.read_csv(TAXONOMY_CSV, dtype={"subtheme_id": str, "theme_id": str})

print(f"  {len(raw)} rows x {len(raw.columns)} columns")
print(f"  Date range: {raw['date'].iloc[0].strftime('%Y-%m-%d')} to "
      f"{raw['date'].iloc[-1].strftime('%Y-%m-%d')}")
print(f"  Taxonomy: {len(meta_full)} features, "
      f"{meta_full['subtheme_id'].nunique()} subthemes, "
      f"{meta_full['theme_id'].nunique()} themes")

# %%
# ── B.2: Identify feature columns ──
#
# No calendar-feature exclusion is applied here (unlike the old notebook):
# the taxonomy is already reconciled against the current, post-exclusion
# feature set (confirmed by the pre-flight check above). If calendar-type
# features are still present under a "calendar" theme/subtheme name, that
# is a deliberate decision already baked into Stage 5's feature set, not
# something this notebook should silently override.

excluded_columns = {"date", "target_daily_return"}
feature_columns = [c for c in raw.columns if c not in excluded_columns]

n_features = len(feature_columns)
print(f"\n  Total columns in parquet: {len(raw.columns)}")
print(f"  Excluded (date, target):  {len(excluded_columns)}")
print(f"  Features retained:        {n_features}")

# %%
# ── B.3: Tag features by theme (NEW taxonomy structure) ──
#
# Panel A/B/C/D tagging is dropped -- see header notes. Theme/subtheme
# structure is the only grouping carried forward.

theme_map    = meta_full.set_index("column")["theme_id"]
subtheme_map = meta_full.set_index("column")["subtheme_id"]

print(f"\n  Themes:    {meta_full['theme_id'].nunique()}")
print(f"  Subthemes: {meta_full['subtheme_id'].nunique()}")

# ==========================================================================
# %% [markdown]
# ## Part C: Compute Target and Verify Alignment
#
# **Target transformation:** the S&P 500 (cap-weighted market) daily
# return is converted to a 21-day rolling cumulative return using the
# log-return method, identical to the old notebook:
#   monthly_return_t = exp( sum_{d=0}^{20} ln(1 + r_{t-d}) ) - 1
#
# **ALIGNMENT CHECK (new):** target_daily_return is the return EARNED ON
# day t (i.e. it is NOT pre-shifted the way some other Stage 5 targets
# are for other models in this project). The rolling sum at row t
# therefore correctly covers days [t-20, t] inclusive -- verified below
# by reconstructing a single window's cumulative return via direct
# compounding and comparing to the vectorised rolling-sum result.

# %%
daily_returns = raw["target_daily_return"].values
log_returns = np.log1p(daily_returns)
monthly_target = np.expm1(
    pd.Series(log_returns).rolling(21, min_periods=21).sum().values
)

print(f"Target (21-day cumulative return):")
print(f"  First valid row: 20 ({raw['date'].iloc[20].strftime('%Y-%m-%d')})")
print(f"  Mean: {np.nanmean(monthly_target):.6f}")
print(f"  Std:  {np.nanstd(monthly_target):.6f}")

# %%
# ── C.1: MANDATORY target alignment check ──
#
# Directly verify the rolling-sum construction against a manual
# compounding calculation for several spot-checked rows, confirming the
# window boundaries are exactly [t-20, t] with no off-by-one error.

print("\n" + "=" * 70)
print("TARGET ALIGNMENT VERIFICATION")
print("=" * 70)

_check_rows = [100, 500, 1500, len(raw) - 100]
_all_aligned = True

for _t in _check_rows:
    if _t < 20 or _t >= len(raw):
        continue
    _window_returns = daily_returns[_t - 20 : _t + 1]
    _manual = np.prod(1 + _window_returns) - 1
    _vectorised = monthly_target[_t]
    _diff = abs(_manual - _vectorised)

    print(f"  Row {_t} ({raw['date'].iloc[_t].strftime('%Y-%m-%d')}): "
          f"manual={_manual:.8f}  vectorised={_vectorised:.8f}  "
          f"diff={_diff:.2e}")

    if _diff > 1e-6:   # was 1e-9 -- too strict for log-space rolling sums
        _all_aligned = False

assert _all_aligned, (
    "Target alignment check FAILED -- manual compounding does not match "
    "the vectorised rolling-sum construction at one or more spot-checked "
    "rows. Do not proceed with a misaligned target. STOP."
)
print(f"\n  ✓ Target alignment verified: rolling window is exactly "
      f"[t-20, t] inclusive, matching manual compounding to <1e-6")

del _check_rows, _all_aligned

# ==========================================================================
# %% [markdown]
# ## Part D: Lag Features by 21 Trading Days
#
# Unchanged from the old notebook: at row t, feature_lagged[t] =
# raw_feature[t - 21]. No rolling average -- raw (already z-scored,
# clipped, NaN-filled) values, lagged only.

# %%
feature_values_raw = raw[feature_columns].values.astype(np.float64)

feature_values = np.full_like(feature_values_raw, np.nan)
feature_values[21:] = feature_values_raw[:-21]

all_dates = raw["date"].values

print(f"Features (lagged by 21 days):")
print(f"  Shape: {feature_values.shape}")
print(f"  First valid row: 21 ({raw['date'].iloc[21].strftime('%Y-%m-%d')})")

# %%
# ── D.1: Feature-target alignment check ──
#
# Confirm row t's lagged feature value genuinely equals row (t-21)'s raw
# feature value -- catches any off-by-one in the shift direction.

_check_col_idx = 0
_check_row = 500
_expected = feature_values_raw[_check_row - 21, _check_col_idx]
_actual = feature_values[_check_row, _check_col_idx]

print(f"\nFeature lag verification (column '{feature_columns[_check_col_idx]}', "
      f"row {_check_row}):")
print(f"  feature_values[{_check_row}]      = {_actual:.6f}")
print(f"  feature_values_raw[{_check_row-21}] = {_expected:.6f}")

assert _actual == _expected, (
    f"Feature lag misaligned: feature_values[{_check_row}] does not equal "
    f"feature_values_raw[{_check_row-21}]. STOP."
)
print(f"  ✓ Confirmed: feature_values[t] == feature_values_raw[t-21]")

del _check_col_idx, _check_row, _expected, _actual

# ==========================================================================
# %% [markdown]
# ## Part E: Determine the Valid Estimation Period

# %%
WINDOW = 1260
first_valid_row = 21

dates_series = pd.Series(range(len(all_dates)), index=pd.DatetimeIndex(all_dates))
month_ends = dates_series.resample("ME").last().dropna().astype(int).values

min_row = first_valid_row + WINDOW - 1
valid_month_ends = month_ends[month_ends >= min_row]

n_months = len(valid_month_ends)
estimation_dates = all_dates[valid_month_ends]

print(f"\nEstimation schedule:")
print(f"  Window size: {WINDOW} days (5 years)")
print(f"  First valid estimation: row {min_row} "
      f"({pd.Timestamp(all_dates[min_row]).strftime('%Y-%m-%d')})")
print(f"  First month-end with full window: "
      f"{pd.Timestamp(estimation_dates[0]).strftime('%Y-%m-%d')}")
print(f"  Last estimation: "
      f"{pd.Timestamp(estimation_dates[-1]).strftime('%Y-%m-%d')}")
print(f"  Total monthly estimation dates: {n_months}")

# ==========================================================================
# %% [markdown]
# ## Part F: LNLM Fitting Functions
#
# UNCHANGED from the old notebook. LNLM for factor i blends a linear
# model and a degree-4 Hermite polynomial model, with mu* found via
# 10-fold stratified cross-validation.

# %%
def stratified_kfold(y, k=10, seed=42):
    n = len(y)
    order = np.argsort(y)
    folds = np.zeros(n, dtype=int)
    rng = np.random.RandomState(seed)

    for start in range(0, n - k + 1, k):
        bucket = order[start:start + k]
        perm = rng.permutation(k)
        for j, idx in enumerate(bucket):
            folds[idx] = perm[j]

    remainder = n % k
    if remainder > 0:
        for j, idx in enumerate(order[n - remainder:]):
            folds[idx] = rng.randint(0, k)

    return folds


def fit_single_factor(x, y_centered, folds, k=10):
    mu_grid = np.linspace(0, 1, 101)

    fold_mus = np.zeros(k)
    fold_xis = np.zeros(k)

    for fold in range(k):
        test_mask = folds == fold
        train_mask = ~test_mask

        x_tr, y_tr = x[train_mask], y_centered[train_mask]
        x_te, y_te = x[test_mask], y_centered[test_mask]

        if len(x_te) < 5 or len(x_tr) < 20:
            continue

        xx = np.dot(x_tr, x_tr)
        if xx < 1e-20:
            continue
        b_lin = np.dot(x_tr, y_tr) / xx

        x2_tr = x_tr * x_tr
        H_tr = np.column_stack([
            x_tr,
            x2_tr - 1,
            x2_tr * x_tr - 3 * x_tr,
            x2_tr * x2_tr - 6 * x2_tr + 3
        ])
        try:
            b_nonlin, _, _, _ = np.linalg.lstsq(H_tr, y_tr, rcond=None)
        except Exception:
            continue

        pred_lin = x_te * b_lin

        x2_te = x_te * x_te
        H_te = np.column_stack([
            x_te,
            x2_te - 1,
            x2_te * x_te - 3 * x_te,
            x2_te * x2_te - 6 * x2_te + 3
        ])
        pred_nonlin = H_te @ b_nonlin

        base_resid = y_te - pred_lin
        diff = pred_nonlin - pred_lin

        resid_matrix = (base_resid[np.newaxis, :]
                        - mu_grid[:, np.newaxis] * diff[np.newaxis, :])
        rmse_vec = np.sqrt(np.mean(resid_matrix ** 2, axis=1))

        best_idx = np.argmin(rmse_vec)
        fold_mus[fold] = mu_grid[best_idx]
        fold_xis[fold] = np.sqrt(np.mean((rmse_vec - rmse_vec[best_idx]) ** 2))

    xi_sum = fold_xis.sum()
    if xi_sum > 1e-10:
        mu_star = np.dot(fold_xis, fold_mus) / xi_sum
    else:
        mu_star = fold_mus.mean()

    xx = np.dot(x, x)
    beta_lin = np.dot(x, y_centered) / xx if xx > 1e-20 else 0.0

    x2 = x * x
    H_all = np.column_stack([
        x, x2 - 1, x2 * x - 3 * x, x2 * x2 - 6 * x2 + 3
    ])
    try:
        beta_nonlin, _, _, _ = np.linalg.lstsq(H_all, y_centered, rcond=None)
    except Exception:
        beta_nonlin = np.zeros(4)

    return mu_star, beta_lin, beta_nonlin

# ==========================================================================
# %% [markdown]
# ## Part G: Fit All Features

# %%
beta_lin_all    = np.full((n_months, n_features), np.nan)
beta_nonlin_all = np.full((n_months, n_features, 4), np.nan)
mu_star_all     = np.full((n_months, n_features), np.nan)
y_mean_all      = np.full(n_months, np.nan)

n_screened = 0

print(f"Fitting LNLM: {n_features} features x {n_months} months")
print(f"  Each fit: 10-fold CV over 101 mu values, then final fit")
print(f"  Estimated time: 30-90 minutes depending on n_features")
print()

t0 = time.perf_counter()

for m in range(n_months):
    idx = valid_month_ends[m]

    window_start = idx - WINDOW + 1
    window_end = idx + 1

    y_window = monthly_target[window_start:window_end]
    X_window = feature_values[window_start:window_end]

    y_mean_val = np.nanmean(y_window)
    y_mean_all[m] = y_mean_val
    y_centered = y_window - y_mean_val

    valid_target = ~np.isnan(y_centered)

    y_for_folds = y_centered[valid_target]
    folds_base = stratified_kfold(y_for_folds, k=10, seed=42)

    full_folds = np.full(WINDOW, -1, dtype=int)
    full_folds[valid_target] = folds_base

    for f in range(n_features):
        x_raw = X_window[:, f]

        valid = ~np.isnan(x_raw) & valid_target

        if valid.sum() < 100:
            n_screened += 1
            continue
        if np.std(x_raw[valid]) < 1e-10:
            n_screened += 1
            continue

        x_v = x_raw[valid]
        y_v = y_centered[valid]
        folds_v = full_folds[valid]

        n_folds_present = len(np.unique(folds_v[folds_v >= 0]))
        if n_folds_present < 5:
            x2 = x_v * x_v
            H = np.column_stack([x_v, x2 - 1, x2 * x_v - 3 * x_v,
                                  x2 * x2 - 6 * x2 + 3])
            try:
                bn, _, _, _ = np.linalg.lstsq(H, y_v, rcond=None)
                mu_star_all[m, f] = 1.0
                beta_lin_all[m, f] = 0.0
                beta_nonlin_all[m, f] = bn
            except Exception:
                pass
            continue

        ms, bl, bn = fit_single_factor(x_v, y_v, folds_v, k=10)
        mu_star_all[m, f] = ms
        beta_lin_all[m, f] = bl
        beta_nonlin_all[m, f] = bn

    elapsed = time.perf_counter() - t0
    if (m + 1) % 10 == 0 or m == 0:
        rate = elapsed / (m + 1)
        eta = rate * (n_months - m - 1)
        pct = (m + 1) / n_months * 100
        print(f"  Month {m+1:3d}/{n_months} ({pct:4.1f}%)  "
              f"[{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]")

total_time = time.perf_counter() - t0
print(f"\nTotal fitting time: {total_time:.0f}s ({total_time/60:.1f} minutes)")
print(f"Near-constant screenings: {n_screened} "
      f"({n_screened / (n_months * n_features):.2%} of all fits)")

# ==========================================================================
# %% [markdown]
# ## Part H: Post-Fit Cleanup

# %%
valid_months_per_feature = np.sum(~np.isnan(mu_star_all), axis=0)

bn_norms = np.linalg.norm(beta_nonlin_all, axis=2)
max_norm_per_feature = np.nanmax(bn_norms, axis=0)

has_inf_lin = np.any(~np.isfinite(beta_lin_all), axis=0)
has_inf_nonlin = np.any(
    ~np.isfinite(beta_nonlin_all.reshape(n_months, n_features, 4)),
    axis=(0, 2)
)

NORM_THRESHOLD = 50
min_valid_months = n_months * 0.5

fails_coverage = valid_months_per_feature < min_valid_months
fails_norm = max_norm_per_feature > NORM_THRESHOLD
fails_finite = has_inf_lin | has_inf_nonlin

is_bad = fails_coverage | fails_norm | fails_finite

n_bad = is_bad.sum()
if n_bad > 0:
    beta_lin_all[:, is_bad] = np.nan
    beta_nonlin_all[:, is_bad, :] = np.nan
    mu_star_all[:, is_bad] = np.nan

print("=" * 70)
print("POST-FIT CLEANUP")
print("=" * 70)
print(f"\n  Total features: {n_features}")

print(f"\n  Criterion 1 -- Valid in >={min_valid_months:.0f}/{n_months} months:")
print(f"    Failed: {fails_coverage.sum()} features")

print(f"\n  Criterion 2 -- Max beta_nonlin norm <= {NORM_THRESHOLD}:")
print(f"    Failed: {fails_norm.sum()} features")

print(f"\n  Criterion 3 -- No inf/NaN in coefficients:")
print(f"    Failed: {fails_finite.sum()} features")

print(f"\n  TOTAL REMOVED: {n_bad} features")
print(f"  REMAINING:     {n_features - n_bad} features with clean parameters")

# Per-theme breakdown (replaces the old per-panel breakdown)
_theme_id_arr = np.array([theme_map.get(c, "UNKNOWN") for c in feature_columns])
for _tid in sorted(set(_theme_id_arr)):
    _theme_mask = _theme_id_arr == _tid
    _n_theme = _theme_mask.sum()
    _n_theme_bad = (_theme_mask & is_bad).sum()
    _n_theme_ok = _n_theme - _n_theme_bad
    _tname = meta_full.loc[meta_full["theme_id"] == _tid, "theme_name"].iloc[0] \
        if (meta_full["theme_id"] == _tid).any() else "?"
    print(f"    Theme {_tid} ({_tname}): {_n_theme_ok}/{_n_theme} clean "
          f"({_n_theme_bad} removed)")

# ==========================================================================
# %% [markdown]
# ## Part I: Excluded Factor Details

# %%
exclusion_reasons = []
for f in range(n_features):
    reasons = []
    name = feature_columns[f]
    tid = theme_map.get(name, "unknown")
    sid = subtheme_map.get(name, "unknown")

    n_valid = valid_months_per_feature[f]
    mn = max_norm_per_feature[f]

    if n_valid < min_valid_months:
        reasons.append(f"low coverage ({n_valid}/{n_months} months)")
    if mn > NORM_THRESHOLD:
        reasons.append(f"explosive norm ({mn:.1f})")
    if has_inf_lin[f]:
        reasons.append("inf in beta_lin")
    if has_inf_nonlin[f]:
        reasons.append("inf in beta_nonlin")

    if len(reasons) > 0:
        exclusion_reasons.append({
            "feature": name, "theme_id": tid, "subtheme_id": sid,
            "n_valid_months": n_valid, "max_norm": mn,
            "reasons": " + ".join(reasons),
        })

excluded_df = pd.DataFrame(exclusion_reasons)

print("=" * 90)
print("EXCLUDED FACTORS -- FULL DETAIL")
print("=" * 90)
print(f"\n  Total excluded: {len(excluded_df)}")
if len(excluded_df) > 0:
    print(f"\n  By theme:")
    for tid in sorted(excluded_df["theme_id"].unique()):
        count = (excluded_df["theme_id"] == tid).sum()
        print(f"    Theme {tid}: {count}")

# %%
surviving_mask_check = ~is_bad
surviving_norms = max_norm_per_feature[surviving_mask_check]
surviving_names = np.array(feature_columns)[surviving_mask_check]

top_norm_idx = np.argsort(surviving_norms)[::-1][:10]
print(f"\n{'=' * 70}")
print(f"BORDERLINE SURVIVORS -- 10 highest beta_nonlin norms among kept factors")
print(f"{'=' * 70}")
print(f"\n  {'Feature':50s} {'Max Norm':>9s}")
for i in top_norm_idx:
    print(f"  {surviving_names[i]:50s} {surviving_norms[i]:9.2f}")

# ==========================================================================
# %% [markdown]
# ## Part J: Parameter Diagnostics and Save

# %%
valid_per_month = np.sum(~np.isnan(mu_star_all), axis=1)
ms_valid = mu_star_all[~np.isnan(mu_star_all)]

print("=" * 70)
print("PARAMETER DIAGNOSTICS")
print("=" * 70)

print(f"\n  Valid features per month: min={valid_per_month.min()}, "
      f"max={valid_per_month.max()}, mean={valid_per_month.mean():.0f}")

print(f"\n  mu* distribution (all valid fits):")
print(f"    Mean:   {ms_valid.mean():.4f}")
print(f"    Median: {np.median(ms_valid):.4f}")
print(f"    > 0.5:  {(ms_valid > 0.5).mean():.1%}")

# %%
output_path = DATA_PM / f"lnlm_params_{DATASET_NAME}.npz"

np.savez(
    output_path,
    beta_lin=beta_lin_all,
    beta_nonlin=beta_nonlin_all,
    mu_star=mu_star_all,
    y_mean=y_mean_all,
    dates=estimation_dates,
    feature_names=np.array(feature_columns),
)

print(f"\nParameters saved to {output_path}")
print(f"  beta_lin:    {beta_lin_all.shape}")
print(f"  beta_nonlin: {beta_nonlin_all.shape}")
print(f"  mu_star:     {mu_star_all.shape}")
print(f"  y_mean:      {y_mean_all.shape}")
print(f"  dates:       {len(estimation_dates)}")
print(f"  features:    {len(feature_columns)}")

# %%
# ── J.1: Save feature metadata (theme/subtheme only, no frequency tag) ──
#
# The old daily/weekly/monthly frequency-tagging heuristic relied on the
# OLD panel naming convention (Panel A/B/C/D prefixes, specific keyword
# lists) which does not carry over to the new feature names. Rather than
# silently mis-tag features under a heuristic built for a different
# naming scheme, this is left out. If frequency tagging is wanted for
# Notebook 2/3 reporting, it should be rebuilt against the new column
# names deliberately, not inherited from this old logic.

meta_out = meta_full[meta_full["column"].isin(feature_columns)].copy()
meta_out.to_csv(DATA_PM / f"polymodel_feature_meta_{DATASET_NAME}.csv", index=False)

print(f"\nFeature metadata saved: {len(meta_out)} rows "
      f"({meta_out['theme_id'].nunique()} themes, "
      f"{meta_out['subtheme_id'].nunique()} subthemes)")

# %%
# ── J.2: Save target and unlagged features ──

target_out = pd.DataFrame({
    "date": raw["date"],
    "monthly_return": monthly_target,
    "daily_return": daily_returns,
})
target_out.to_parquet(DATA_PM / f"polymodel_target_{DATASET_NAME}.parquet", index=False)
print(f"\nTarget saved: {target_out.shape}")

features_out = raw[["date"] + feature_columns].copy()
features_out.to_parquet(DATA_PM / f"polymodel_features_unlagged_{DATASET_NAME}.parquet", index=False)
print(f"Features (unlagged) saved: {features_out.shape}")

print(f"\n{'=' * 70}")
print("COMPLETE")
print(f"  Parameters: {output_path}")
print(f"  Metadata:   {DATA_PM / f'polymodel_feature_meta_{DATASET_NAME}.csv'}")
print(f"  Target:     {DATA_PM / f'polymodel_target_{DATASET_NAME}.parquet'}")
print(f"  Features:   {DATA_PM / f'polymodel_features_unlagged_{DATASET_NAME}.parquet'}")
print(f"{'=' * 70}")

DATASET_NAME:     agg_means
Feature parquet:  ..\..\Data\Data_Collection\Final\Stage_5_Model_Ready\02_assembled\agg_means.parquet
Taxonomy CSV:     ..\..\Data\Data_Collection\Final\Stage_5_Model_Ready\05_themes\numbered_classified_moment_inventory_means_only.csv

PRE-FLIGHT VERIFICATION

  [1] target_daily_return present: YES
      NaN count: 0

  [2] Total NaN in feature columns: 0

  [3] Max |feature value|: 5.0000
      Values with |z| > 5: 0

  [4] Binary regime indicators present: []
      ✓ Confirmed clean -- no binaries present

  [5] Date range: 2007-08-01 -> 2024-12-30
      Total rows: 4384
      Total feature columns: 574

  [6] Taxonomy reconciliation:
      Features in parquet:        574
      Features in taxonomy CSV:   574
      In features, NOT taxonomy:  0
      In taxonomy, NOT features:  0
      ✓ Exact 1:1 reconciliation confirmed -- every feature has exactly one taxonomy row, every taxonomy row has exactly one feature

  [7] Duplicate taxonomy entries: 0
      ✓ N